In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

folder = "/content/drive/MyDrive/P1_Airline"

# Peek at 2025 file - just 5 rows, no RAM used
df_peek = pd.read_csv(f"{folder}/bts_ontime_2025.csv", nrows=5)

print("Total columns:", len(df_peek.columns))
print("\nAll column names:")
for col in df_peek.columns.tolist():
    print(f"  {col}")

print("\nSample data:")
df_peek.head()

Mounted at /content/drive
Total columns: 38

All column names:
  Year
  Quarter
  Month
  DayofMonth
  DayOfWeek
  FlightDate
  Reporting_Airline
  Tail_Number
  Flight_Number_Reporting_Airline
  Origin
  OriginCityName
  OriginState
  OriginStateName
  Dest
  DestCityName
  DestState
  DestStateName
  CRSDepTime
  DepTime
  DepDelay
  DepDelayMinutes
  CRSArrTime
  ArrTime
  ArrDelay
  ArrDelayMinutes
  Cancelled
  CancellationCode
  Diverted
  CRSElapsedTime
  ActualElapsedTime
  AirTime
  Flights
  Distance
  CarrierDelay
  WeatherDelay
  NASDelay
  SecurityDelay
  LateAircraftDelay

Sample data:


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,CRSElapsedTime,ActualElapsedTime,AirTime,Flights,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,1,1,1,3,2025-01-01,AA,N104NN,1,JFK,...,381.0,377.0,345.0,1.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,N110AN,1,JFK,...,381.0,390.0,353.0,1.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,N106NN,1,JFK,...,381.0,371.0,347.0,1.0,2475.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,N117AN,1,JFK,...,386.0,383.0,349.0,1.0,2475.0,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,N104NN,1,JFK,...,381.0,378.0,347.0,1.0,2475.0,NaN,NaN,NaN,NaN,NaN


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3
import os

folder = "/content/drive/MyDrive/P1_Airline"
db_path = f"{folder}/airline_db.sqlite"

print("✅ Libraries loaded")
print(f"📁 Folder: {folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries loaded
📁 Folder: /content/drive/MyDrive/P1_Airline


In [3]:
# We only keep 20 of the 38 columns — saves RAM significantly
COLS = [
    'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',
    'Origin', 'OriginCityName', 'OriginState',
    'Dest', 'DestCityName', 'DestState',
    'DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes',
    'Cancelled', 'CancellationCode', 'Diverted',
    'AirTime', 'Distance',
    'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay'
]

FILES = [
    "bts_ontime_2024.csv",
    "bts_ontime_2025.csv",
    "bts_ontime_2026.csv"
]

print("✅ Config ready")
print(f"📊 Keeping {len(COLS)} columns out of 38")

✅ Config ready
📊 Keeping 23 columns out of 38


In [4]:
CHUNK_SIZE = 100_000  # 100k rows at a time — safe for Colab

all_chunks = []
total_rows = 0

for filename in FILES:
    filepath = f"{folder}/{filename}"
    print(f"\n📂 Loading {filename}...")

    file_rows = 0
    for chunk in pd.read_csv(filepath, usecols=COLS, chunksize=CHUNK_SIZE, low_memory=False):
        all_chunks.append(chunk)
        file_rows += len(chunk)

    total_rows += file_rows
    print(f"   ✅ {file_rows:,} rows loaded")

df = pd.concat(all_chunks, ignore_index=True)
del all_chunks  # free RAM immediately

print(f"\n🎯 Total rows across all files: {total_rows:,}")
print(f"📐 DataFrame shape: {df.shape}")


📂 Loading bts_ontime_2024.csv...
   ✅ 7,079,061 rows loaded

📂 Loading bts_ontime_2025.csv...
   ✅ 7,001,619 rows loaded

📂 Loading bts_ontime_2026.csv...
   ✅ 544,003 rows loaded

🎯 Total rows across all files: 14,624,683
📐 DataFrame shape: (14624683, 23)


In [5]:
print("🧹 Cleaning data...\n")

# 1. Rename columns to clean names
df.rename(columns={
    'Reporting_Airline': 'carrier_code',
    'Flight_Number_Reporting_Airline': 'flight_number',
    'FlightDate': 'flight_date',
    'Origin': 'origin',
    'OriginCityName': 'origin_city',
    'OriginState': 'origin_state',
    'Dest': 'dest',
    'DestCityName': 'dest_city',
    'DestState': 'dest_state',
    'DepDelay': 'dep_delay',
    'DepDelayMinutes': 'dep_delay_min',
    'ArrDelay': 'arr_delay',
    'ArrDelayMinutes': 'arr_delay_min',
    'Cancelled': 'cancelled',
    'CancellationCode': 'cancellation_code',
    'Diverted': 'diverted',
    'AirTime': 'air_time',
    'Distance': 'distance',
    'CarrierDelay': 'carrier_delay',
    'WeatherDelay': 'weather_delay',
    'NASDelay': 'nas_delay',
    'SecurityDelay': 'security_delay',
    'LateAircraftDelay': 'late_aircraft_delay'
}, inplace=True)

# 2. Fix date column
df['flight_date'] = pd.to_datetime(df['flight_date'])

# 3. Fill NaN delays with 0 (no delay = 0 minutes)
delay_cols = ['dep_delay', 'dep_delay_min', 'arr_delay', 'arr_delay_min',
              'carrier_delay', 'weather_delay', 'nas_delay',
              'security_delay', 'late_aircraft_delay']
df[delay_cols] = df[delay_cols].fillna(0)

# 4. Fill cancelled/diverted with 0
df['cancelled'] = df['cancelled'].fillna(0).astype(int)
df['diverted'] = df['diverted'].fillna(0).astype(int)

# 5. Fill cancellation code with 'N' (not cancelled)
df['cancellation_code'] = df['cancellation_code'].fillna('N')

# 6. Add flight_id primary key
df.insert(0, 'flight_id', range(1, len(df) + 1))

# 7. Remove rows with missing origin or dest (can't use these)
before = len(df)
df.dropna(subset=['origin', 'dest', 'carrier_code'], inplace=True)
after = len(df)

print(f"✅ Columns renamed")
print(f"✅ Dates fixed")
print(f"✅ NaN delays filled with 0")
print(f"✅ Removed {before - after:,} rows with missing origin/dest/carrier")
print(f"✅ Final row count: {after:,}")
print(f"\n📋 Null check after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

🧹 Cleaning data...

✅ Columns renamed
✅ Dates fixed
✅ NaN delays filled with 0
✅ Removed 0 rows with missing origin/dest/carrier
✅ Final row count: 14,624,683

📋 Null check after cleaning:
flight_number         1
air_time         262730
dtype: int64


In [6]:
print("🏗️ Building dimension tables...\n")

# AIRPORTS table (unique airports from both origin and dest)
origins = df[['origin', 'origin_city', 'origin_state']].rename(columns={
    'origin': 'airport_code', 'origin_city': 'city', 'origin_state': 'state'
})
dests = df[['dest', 'dest_city', 'dest_state']].rename(columns={
    'dest': 'airport_code', 'dest_city': 'city', 'dest_state': 'state'
})
airports = pd.concat([origins, dests]).drop_duplicates(subset='airport_code').reset_index(drop=True)
print(f"✅ AIRPORTS table: {len(airports):,} unique airports")

# CARRIERS table (unique airlines)
carriers = df[['carrier_code']].drop_duplicates().reset_index(drop=True)
carriers['carrier_name'] = carriers['carrier_code']  # BTS has no full name in this dataset
print(f"✅ CARRIERS table: {len(carriers):,} unique carriers")

# FLIGHTS fact table
flights = df[[
    'flight_id', 'carrier_code', 'flight_number', 'flight_date',
    'origin', 'dest', 'dep_delay_min', 'arr_delay_min',
    'cancelled', 'cancellation_code', 'diverted', 'air_time', 'distance'
]].copy()
print(f"✅ FLIGHTS table: {len(flights):,} flights")

# DELAYS table
delays = df[[
    'flight_id', 'carrier_delay', 'weather_delay',
    'nas_delay', 'security_delay', 'late_aircraft_delay'
]].copy()
delays.insert(0, 'delay_id', range(1, len(delays) + 1))
print(f"✅ DELAYS table: {len(delays):,} rows")

🏗️ Building dimension tables...

✅ AIRPORTS table: 358 unique airports
✅ CARRIERS table: 15 unique carriers
✅ FLIGHTS table: 14,624,683 flights
✅ DELAYS table: 14,624,683 rows


In [8]:
# In Colab — store credentials as variables, never hardcode in code
# Use Colab's secret manager or just input() for now

DB_HOST = "ep-super-cell-aofus8nt-pooler.c-2.ap-southeast-1.aws.neon.tech"
DB_NAME = "neondb"
DB_USER = "neondb_owner"
DB_PASSWORD = input("Paste your DB password here:  ")  # you type it, never saved

conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}?sslmode=require"

Paste your DB password here:  npg_8GWDIsBC5PkM
